<a href="https://colab.research.google.com/github/caramos84/QC_Video/blob/main/AudioAnalyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QC Video - Notebook 03
## Audio Analyzer

Este notebook analiza la pista de audio generada por el Notebook 01 (Asset Decomposer).

Objetivos

- Detectar presencia de audio
- Transcribir voz mediante Whisper
- Detectar idioma
- Contar palabras
- Generar transcript
- Generar análisis de audio

Entrada

output.zip

Salida

output/
│
├── transcript.txt
├── transcript.json
└── audio_analysis.json

In [1]:
# Instalación

!apt-get update -qq
!apt-get install ffmpeg -y

!pip install -q openai-whisper
!pip install -q ffmpeg-python
!pip install -q pandas

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 71 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.1 MB/s eta 0:00:00


## Importación de librerías

In [2]:
import os
import json
import shutil
import zipfile

import whisper
import pandas as pd

from google.colab import files

## Cargar paquete generado por Notebook 01

In [3]:
uploaded = files.upload()

zip_files = [
    f
    for f in uploaded.keys()
    if f.endswith(".zip")
]

if len(zip_files) == 0:
    raise Exception("Debe subir output.zip")

ZIP_PATH = zip_files[0]

print(ZIP_PATH)

Saving output.zip to output.zip
output.zip


## Descomprimir paquete

In [4]:
OUTPUT_DIR="/content/output"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

with zipfile.ZipFile(ZIP_PATH,"r") as z:
    z.extractall("/content")

print("Output restaurado")

Output restaurado


## Validar estructura

In [5]:
AUDIO_PATH=os.path.join(
    OUTPUT_DIR,
    "audio.wav"
)

METADATA_PATH=os.path.join(
    OUTPUT_DIR,
    "metadata.json"
)

assert os.path.exists(AUDIO_PATH)
assert os.path.exists(METADATA_PATH)

print("Audio encontrado")

Audio encontrado


## Cargar metadata

In [7]:
with open(METADATA_PATH) as f:

    metadata=json.load(f)

metadata

{'filename': 'OFERTA 1 ESCOLAR 6.01.26_V3.mp4',
 'duration': 10.01,
 'fps': 29.97002997002997,
 'width': 1920,
 'height': 1080,
 'orientation': 'horizontal',
 'audio_present': True}

## Cargar modelo Whisper

In [6]:
model = whisper.load_model("base")

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 79.2MiB/s]


## Transcripción

In [8]:
result = model.transcribe(
    AUDIO_PATH
)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


## Guardar transcript.txt

In [9]:
TRANSCRIPT_TXT=os.path.join(
    OUTPUT_DIR,
    "transcript.txt"
)

with open(
    TRANSCRIPT_TXT,
    "w",
    encoding="utf8"
) as f:

    f.write(
        result["text"]
    )

## Guardar transcript.json

In [10]:
TRANSCRIPT_JSON=os.path.join(
    OUTPUT_DIR,
    "transcript.json"
)

with open(
    TRANSCRIPT_JSON,
    "w",
    encoding="utf8"
) as f:

    json.dump(
        result,
        f,
        indent=4,
        ensure_ascii=False
    )

## Calcular métricas

In [11]:
text=result["text"]

words=text.split()

audio_analysis={

    "language":result["language"],

    "duration":metadata["duration"],

    "audio_present":metadata["audio_present"],

    "speech_detected":len(words)>0,

    "word_count":len(words),

    "segments":len(result["segments"])
}

audio_analysis

{'language': 'es',
 'duration': 10.01,
 'audio_present': True,
 'speech_detected': True,
 'word_count': 32,
 'segments': 4}

## Guardar audio_analysis.json

In [12]:
AUDIO_ANALYSIS=os.path.join(
    OUTPUT_DIR,
    "audio_analysis.json"
)

with open(
    AUDIO_ANALYSIS,
    "w"
) as f:

    json.dump(
        audio_analysis,
        f,
        indent=4
    )

## Mostrar transcripción

In [13]:
print(result["text"])

 Con tu éxito, dile ola al regreso a clases. Cuadernos con éstikers desde 3.200 pesos y hasta 50% de descuento en referencia seleccionadas para primo. Of course, 7-7, VELES, SANEVEN y Fiori.


## Tabla de segmentos

In [14]:
segments=[]

for s in result["segments"]:

    segments.append({

        "inicio":s["start"],

        "fin":s["end"],

        "texto":s["text"]

    })

df=pd.DataFrame(segments)

df

,inicio,fin,texto
0,0.00,2.16,"Con tu éxito, dile ola al regreso a clases."
1,2.16,6.88,Cuadernos con éstikers desde 3.200 pesos y ha...
2,6.88,7.88,para primo.
3,7.88,9.96,"Of course, 7-7, VELES, SANEVEN y Fiori."


## Exportar resultados

In [15]:
ZIP_OUTPUT="output_audio_analysis.zip"

if os.path.exists(ZIP_OUTPUT):
    os.remove(ZIP_OUTPUT)

!zip -r output_audio_analysis.zip output > /dev/null

files.download(ZIP_OUTPUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>